# 🧙‍♂️ VYAS-Daivagya-1.5B: The World's First Dedicated Vedic Astrology Foundation Model
### Fast 1-Click Fine-Tuning on Free Google Colab (T4 / V100 / A100 GPU)

This notebook trains **Qwen-2.5-1.5B-Instruct** (or Llama-3.2-3B) on the **VYAS-Daivagya SFT Dataset** (7,500+ classical shastric pairs covering Parashari, Pushkar Navamsha, KP, Jaimini, Nadi, and Lal Kitab remedies).

**Estimated Training Time:** ~35 to 45 minutes on Colab Free T4 GPU.
**Output:** Standalone 1.1 GB GGUF Model (`vyas-daivagya-1.5b-q4_k_m.gguf`) for offline use in VYAS.

In [ ]:
# Step 1: Install Unsloth & High-Performance Fine-Tuning Tools (5x faster, 80% less VRAM)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers
print("✅ Setup Complete!")

In [ ]:
# Step 2: Load Base Model with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect Float16 / Bfloat16
load_in_4bit = True

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {model_name} in 4-bit...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Base Model Loaded!")

In [ ]:
# Step 3: Configure LoRA Adapter Matrix
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("✅ LoRA Adapters Initialized!")

In [ ]:
# Step 4: Load VYAS-Daivagya SFT Dataset
from datasets import load_dataset

# Upload your 'vyas_daivagya_sft_train.jsonl' to Colab or fetch directly
dataset = load_dataset("json", data_files="vyas_daivagya_sft_train.jsonl", split="train")

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples['messages']:
        formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(formatted)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Loaded {len(dataset)} training samples!")

In [ ]:
# Step 5: Execute Training (Takes ~35-40 mins on Colab Free T4!)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("🚀 Starting Training Execution...")
trainer_stats = trainer.train()
print("🎉 Training Complete!")

In [ ]:
# Step 6: Test Inference Instantly
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "आप आचार्य व्यास हैं—महर्षि पराशर, जैमिनी, एवं भृगु ज्योतिष के मूर्धन्य वैदिक विद्वान।"},
    {"role": "user", "content": "मेरी कुण्डली में सूर्य 7° वृश्चिक राशि में है, क्या यह पुष्कर नवांश में है?"}
]
inputs = tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 512, use_cache = True)

In [ ]:
# Step 7: Export to 1.1 GB GGUF for Offline Use in VYAS App!
model.save_pretrained_gguf("vyas-daivagya-1.5b", tokenizer, quantization_method = "q4_k_m")
print("✅ Export Complete! Download 'vyas-daivagya-1.5b-q4_k_m.gguf' (~1.1 GB) from the files sidebar!")